## 1. Data Loading and Initial Overview

The Online Retail II dataset (UCI/Kaggle) — real transactions from a UK-based online retailer, 2009-2011.
This data is genuinely messy: missing values, duplicates, cancelled orders, and administrative/service entries.
We start with a basic structural overview.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\77017\.junie\online_retail_II.csv")
print(df.head())
print(df.shape)


  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
(1067371, 8)


Checking data types and the number of missing values per column.

In [2]:
print(df.dtypes)
print(df.isna().sum())

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


## 2. Exploring the "Dirt" in the Data

Before removing anything, we investigate what types of anomalies exist in the dataset and what they actually mean. Counting cancellations (Quantity < 0), invoices with a 'C' prefix (per UCI documentation — a marker of cancellation), and fully duplicated rows.

In [3]:
cancelsd = len(df[df['Quantity'] < 0])
print(cancelsd)

print(df['Invoice'].str.startswith('C').sum())
print(df.duplicated().sum())

22950
19494
34335


Testing a hypothesis: do rows with negative Quantity match rows with an Invoice starting with 'C'? Checking the mismatch — negative-quantity rows WITHOUT the 'C' prefix.

In [4]:
mask = (df['Quantity'] < 0) & (~df['Invoice'].str.startswith('C'))
df[mask].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


Inspecting full row duplicates manually — are these legitimate repeated line items or a technical export error

In [5]:
dupes = df[df.duplicated(keep=False)].sort_values('Invoice').head(10)
print(dupes)

    Invoice StockCode                        Description  Quantity  \
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
394  489517     21912           VINTAGE SNAKES & LADDERS         1   
391  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
390  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
388  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
386  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   

             InvoiceDate  Price  Customer ID         Country  
362  2009-12-01 11:34:00   3.75      16329.0  United Kingdom  
394  2009-12-01 11:34:00   3.75      16329.0  United Kingdom  
391  2009-12-01 11:34:00   1.95      163

Checking how many cancelled orders (Invoice starting with 'C') have no associated Customer ID — relevant for the later RFM analysis.

In [6]:
cancels = df[df['Invoice'].str.startswith('C')]
print(cancels['Customer ID'].isna().sum())
print(len(cancels))

750
19494


## 3. Data Cleaning

### 3.1 Removing full duplicate rows
The duplicates found (complete matches across all columns) point to a technical export error rather than legitimate repeat purchases. Dropping them.

In [7]:
df_clean = df.drop_duplicates()
print(df.shape)
print(df_clean.shape)

(1067371, 8)
(1033036, 8)


### 3.2 Warehouse adjustment records (damaged/lost stock)
Rows with negative Quantity but WITHOUT a 'C' prefix turned out to be warehouse records (Description: "lost", "damages", etc.), with Price=0 and no Customer ID. Verifying the hypothesis across the full dataset before removal.

In [8]:
stock_mask = (df_clean['Quantity'] < 0) & (~df_clean['Invoice'].str.startswith('C'))
df_clean[stock_mask]['Price'].unique()

array([0.])

Hypothesis confirmed (Price is strictly 0 for all matching rows) — removing warehouse adjustment records.

In [9]:
df_clean = df_clean[~stock_mask]
df_clean.shape

(1029643, 8)

### 3.3 Fixing data types
InvoiceDate was stored as a string — converting to datetime for later calculations (Recency, cohort analysis).

In [10]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

### 3.4 Checking the Price column for anomalies
Found negative and abnormally large values — investigating each case separately.

In [11]:
df_clean['Price'].describe()
df_clean[df_clean['Price'] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


Rows with negative prices turned out to be accounting entries ("Adjust bad debt") with an 'A' invoice prefix — not real sales. Checking the scale of the issue.

In [12]:
bad_debt_mask = df_clean['Invoice'].str.startswith('A')
bad_debt_mask.sum()

np.int64(6)

Removing bad debt adjustment records. Re-checking Price statistics.

In [13]:
df_clean = df_clean[~bad_debt_mask]
df_clean['Price'].describe()

count    1.029637e+06
mean     4.772576e+00
std      9.234914e+01
min      0.000000e+00
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64

Zero-priced rows remain on otherwise normal products — likely free samples/promotions. 
Decision: don't drop these from the main dataframe, filter them out selectively depending on the task (e.g., exclude from revenue analysis, but keep in the full dataset).

In [14]:
print((df_clean['Price'] == 0).sum())
print(df_clean[df_clean['Price'] == 0].head(10))



2621
     Invoice StockCode          Description  Quantity         InvoiceDate  \
3161  489659     21350                  NaN       230 2009-12-01 17:39:00   
3731  489781     84292                  NaN        17 2009-12-02 11:45:00   
4674  489825     22076   6 RIBBONS EMPIRE          12 2009-12-02 13:34:00   
5904  489861       DOT       DOTCOM POSTAGE         1 2009-12-02 14:50:00   
6378  489882    35751C                  NaN        12 2009-12-02 16:22:00   
6555  489898    79323G                  NaN       954 2009-12-03 09:40:00   
6581  489903     21166                  NaN        48 2009-12-03 09:57:00   
6781  489998     48185  DOOR MAT FAIRY CAKE         2 2009-12-03 11:19:00   
7204  490015     21982                  NaN       467 2009-12-03 12:29:00   
9249  490123    84508B                  NaN       184 2009-12-03 18:08:00   

      Price  Customer ID         Country  
3161    0.0          NaN  United Kingdom  
3731    0.0          NaN  United Kingdom  
4674    0.0      1

Checking the highest-priced rows — looking for more hidden service records rather than real products.

In [ ]:
pd.set_option('display.max_columns', None)
print(df_clean.sort_values('Price', ascending=False).head(10))

None
         Invoice     StockCode   Description  Quantity         InvoiceDate  \
748142   C556445             M        Manual        -1 2011-06-10 15:31:00   
241827    512771             M        Manual         1 2010-06-17 16:53:00   
241824   C512770             M        Manual        -1 2010-06-17 16:52:00   
320581   C520667  BANK CHARGES  Bank Charges        -1 2010-08-27 13:42:00   
1050063  C580605     AMAZONFEE    AMAZON FEE        -1 2011-12-05 11:36:00   
569163   C540117     AMAZONFEE    AMAZON FEE        -1 2011-01-05 09:55:00   
569164   C540118     AMAZONFEE    AMAZON FEE        -1 2011-01-05 09:57:00   
519294   C537651     AMAZONFEE    AMAZON FEE        -1 2010-12-07 15:49:00   
517955    537632     AMAZONFEE    AMAZON FEE         1 2010-12-07 15:08:00   
517953   C537630     AMAZONFEE    AMAZON FEE        -1 2010-12-07 15:04:00   

            Price  Customer ID         Country  
748142   38970.00      15098.0  United Kingdom  
241827   25111.09          NaN  United

### 3.5 Finding service StockCodes
Real product codes always contain digits. Searching for StockCodes with no digits at all — candidates for service records (postage, fees, manual adjustments).

In [16]:
service_codes = df_clean[~df_clean['StockCode'].str.contains(r'\d')]['StockCode'].unique()
service_codes

<StringArray>
[        'POST',            'D',          'DOT',            'M',
 'BANK CHARGES',         'PADS',       'ADJUST',            'm',
            'S',     'DCGSSBOY',    'DCGSSGIRL',    'AMAZONFEE',
         'CRUK']
Length: 13, dtype: str

Checking the description behind each service code found, to avoid accidentally removing a real product with a non-standard code (e.g. PADS is an actual product despite its letter-only code).

In [17]:
for code in service_codes:
    print(code, '→', df_clean[df_clean['StockCode'] == code]['Description'].unique()[:1])

POST → <StringArray>
['POSTAGE']
Length: 1, dtype: str
D → <StringArray>
['Discount']
Length: 1, dtype: str
DOT → <StringArray>
['DOTCOM POSTAGE']
Length: 1, dtype: str
M → <StringArray>
['Manual']
Length: 1, dtype: str
BANK CHARGES → <StringArray>
[' Bank Charges']
Length: 1, dtype: str
PADS → <StringArray>
['PADS TO MATCH ALL CUSHIONS']
Length: 1, dtype: str
ADJUST → <StringArray>
['Adjustment by john on 26/01/2010 16']
Length: 1, dtype: str
m → <StringArray>
['Manual']
Length: 1, dtype: str
S → <StringArray>
['SAMPLES']
Length: 1, dtype: str
DCGSSBOY → <StringArray>
['update']
Length: 1, dtype: str
DCGSSGIRL → <StringArray>
['update']
Length: 1, dtype: str
AMAZONFEE → <StringArray>
['AMAZON FEE']
Length: 1, dtype: str
CRUK → <StringArray>
['CRUK Commission']
Length: 1, dtype: str


Separately checking ambiguous codes (DCGSSBOY/DCGSSGIRL contain both service "update" entries and real party bag sales).

In [18]:
for code in ['S', 'DCGSSBOY', 'DCGSSGIRL', 'AMAZONFEE']:
    print(code, '→', df_clean[df_clean['StockCode'] == code]['Description'].unique())

S → <StringArray>
['SAMPLES']
Length: 1, dtype: str
DCGSSBOY → <StringArray>
['update', 'BOYS PARTY BAG']
Length: 2, dtype: str
DCGSSGIRL → <StringArray>
['update', 'GIRLS PARTY BAG']
Length: 2, dtype: str
AMAZONFEE → <StringArray>
['AMAZON FEE']
Length: 1, dtype: str


Final removal of all confirmed service records: postage, bank charges, manual adjustments, Amazon/CRUK fees, and the specific "update" entries.

In [19]:
codes_to_remove = ['POST','D','DOT','M','m','BANK CHARGES','ADJUST','CRUK','S','AMAZONFEE']
mask1 = df_clean['StockCode'].isin(codes_to_remove)
mask2 = (df_clean['StockCode'].isin(['DCGSSBOY','DCGSSGIRL'])) & (df_clean['Description'] == 'update')

df_clean = df_clean[~(mask1 | mask2)]
print(df_clean.shape)
df_clean['Price'].describe()

(1024239, 8)


count    1.024239e+06
mean     3.368151e+00
std      4.957287e+00
min      0.000000e+00
25%      1.250000e+00
50%      2.100000e+00
75%      4.130000e+00
max      1.157150e+03
Name: Price, dtype: float64

### 3.6 Final data quality check
A checkpoint: dataset size, remaining missing values, Price and Quantity statistics after all removals.

In [20]:
print(df_clean.shape)
print(df_clean.isna().sum())
print(df_clean['Price'].describe())
print(df_clean['Quantity'].describe())

(1024239, 8)
Invoice             0
StockCode           0
Description      1633
Quantity            0
InvoiceDate         0
Price               0
Customer ID    229722
Country             0
dtype: int64
count    1.024239e+06
mean     3.368151e+00
std      4.957287e+00
min      0.000000e+00
25%      1.250000e+00
50%      2.100000e+00
75%      4.130000e+00
max      1.157150e+03
Name: Price, dtype: float64
count    1.024239e+06
mean     1.070532e+01
std      1.712027e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64


Investigating quantity outliers (|Quantity| > 5000) — checking whether they're data errors or legitimate large wholesale orders (and their cancellations).

In [21]:
df_clean[df_clean['Quantity'].abs() > 5000]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
7303,490018,21967,PACK OF 12 SKULL TISSUES,5184,2009-12-03 12:31:00,0.25,17940.0,United Kingdom
65077,495194,37410,BLACK AND WHITE PAISLEY FLOWER MUG,6012,2010-01-21 15:11:00,0.10,13902.0,Denmark
65079,495194,16044,POP-ART FLUORESCENT PENS,6144,2010-01-21 15:11:00,0.06,13902.0,Denmark
65085,495194,20759,CHRYSANTHEMUM POCKET BOOK,5280,2010-01-21 15:11:00,0.10,13902.0,Denmark
65088,495194,20756,GREEN FERN POCKET BOOK,5280,2010-01-21 15:11:00,0.10,13902.0,Denmark
65090,495194,20991,JAZZ HEARTS MAGNETIC MEMO PAD,6768,2010-01-21 15:11:00,0.10,13902.0,Denmark
65091,495194,20993,JAZZ HEARTS MEMO PAD,9312,2010-01-21 15:11:00,0.10,13902.0,Denmark
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark
90866,497946,47503E,ASS FLORAL PRINT SCISSORS,6696,2010-02-15 11:57:00,0.15,13902.0,Denmark
93677,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,2010-02-17 10:51:00,0.30,13902.0,Denmark


Confirming Quantity's data type stayed int64 throughout — verifying an earlier assumption rather than guessing.

In [22]:
df_clean.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

## 4. RFM Segmentation

Preparing a task-specific subset: removing rows without a Customer ID (required for grouping by customer) and cancelled orders (RFM should reflect actual completed purchases). Adding a `Total Price` column as the basis for the Monetary metric.

In [23]:
rfm_mask = (~df_clean['Customer ID'].isna()) & (~df_clean['Invoice'].str.startswith('C'))
df_rfm = df_clean[rfm_mask]
df_rfm['Total Price'] = df_rfm['Quantity'] * df_rfm['Price']
print(df_rfm.head(10))

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
5  489434     22064           PINK DOUGHNUT TRINKET POT         24   
6  489434     21871                  SAVE THE PLANET MUG        24   
7  489434     21523   FANCY FONT HOME SWEET HOME DOORMAT        10   
8  489435     22350                            CAT BOWL         12   
9  489435     22349       DOG BOWL , CHASING BALL DESIGN        12   

          InvoiceDate  Price  Customer ID         Country  Total Price  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom         83.4  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom         81.0  
2 2009-12-

Calculating the three RFM metrics per customer: Recency (days since last purchase), Frequency (number of unique orders), Monetary (total amount spent).

In [24]:
snapshot_date = df_rfm['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_rfm.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'Total Price': 'sum'
})

rfm = rfm.rename(columns = {
    'InvoiceDate' : 'Recency',
    'Invoice' : 'Frequency',
    'Total Price' : 'Monetary'
})
print(rfm.head())
print(rfm.describe())

             Recency  Frequency  Monetary
Customer ID                              
12346.0          326         12  77556.46
12347.0            2          8   4921.53
12348.0           75          5   1658.40
12349.0           19          3   3678.69
12350.0          310          1    294.40
           Recency    Frequency       Monetary
count  5856.000000  5856.000000    5856.000000
mean    200.376025     6.253074    2917.022846
std     208.665940    12.761324   14331.987029
min       1.000000     1.000000       0.000000
25%      25.000000     1.000000     340.457500
50%      95.000000     3.000000     855.860000
75%     379.000000     7.000000    2238.582500
max     739.000000   375.000000  580987.040000


Checking the edge case of customers with Monetary = 0 (free-sample-only orders) — negligible share, safe to ignore. Splitting each metric into quartile-based scores (1-4) and combining them into a single RFM code string.

In [25]:
print(rfm[rfm['Monetary'] == 0])

rfm['R_score'] = pd.qcut(rfm['Recency'], q=4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'], q=4, labels=[1,2,3], duplicates='drop')
rfm['M_score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1,2,3,4])

rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

rfm.head()

             Recency  Frequency  Monetary
Customer ID                              
13256.0           14          1       0.0
14103.0          665          1       0.0
14827.0          665          1       0.0


,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score
Customer ID,,,,,,,
12346.0,326,12,77556.46,2,3,4,234
12347.0,2,8,4921.53,4,3,4,434
12348.0,75,5,1658.40,3,2,3,323
12349.0,19,3,3678.69,4,1,4,414
12350.0,310,1,294.40,2,1,1,211


Mapping RFM score combinations to business-friendly segment labels: Champions, At Risk, New customers, Lost, Other.

In [26]:
def segment(row):
    if row['R_score'] >= 3 and row['F_score'] >= 2 and row['M_score'] >= 3:
        return 'Champions'
    elif row['R_score'] <= 2 and row['F_score'] >= 2 and row['M_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] >= 3 and row['F_score'] <= 2 and row['M_score'] <= 2:
        return 'New customers'
    elif row['R_score'] <= 2 and row['F_score'] <= 2 and row['M_score'] <= 2:
        return 'Lost'
    else:
        return 'Other'
    
rfm['Segment'] = rfm.apply(segment, axis=1)
rfm['Segment'].value_counts()


Segment
Lost             1994
Champions        1746
New customers     919
Other             610
At Risk           587
Name: count, dtype: int64

## 5. Cohort Retention Analysis

Preparing a task-specific subset (same filtering logic as for RFM: no missing Customer ID, no cancelled orders) to analyze customer retention over time.

In [27]:
cohort_mask = (~df_clean['Customer ID'].isna()) & (~df_clean['Invoice'].str.startswith('C'))
df_cohort = df_clean[cohort_mask]
print(df_cohort.head())



  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


Determining each customer's cohort (month of first purchase) and calculating the Cohort Index — how many months have passed since that customer's first purchase, for each transaction.

In [28]:
df_cohort['InvoiceMonth'] = df_cohort['InvoiceDate'].dt.to_period('M')
df_cohort['CohortMonth'] = df_cohort.groupby('Customer ID')['InvoiceMonth'].transform('min')
df_cohort['CohortIndex'] = (df_cohort['InvoiceMonth'] - df_cohort['CohortMonth']).apply(lambda x: x.n)
sample_id = df_cohort['Customer ID'].iloc[100] 
df_cohort[df_cohort['Customer ID'] == sample_id][['Customer ID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']]


,Customer ID,InvoiceMonth,CohortMonth,CohortIndex
96,13635.0,2009-12,2009-12,0
97,13635.0,2009-12,2009-12,0
98,13635.0,2009-12,2009-12,0
99,13635.0,2009-12,2009-12,0
100,13635.0,2009-12,2009-12,0
...,...,...,...,...
898415,13635.0,2011-10,2009-12,22
898416,13635.0,2011-10,2009-12,22
898417,13635.0,2011-10,2009-12,22
898418,13635.0,2011-10,2009-12,22


Building the retention matrix: rows = cohorts (first purchase month), columns = cohort index (months since start), values = number of unique returning customers.

In [29]:
cohort_data = df_cohort.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='Customer ID')

print(cohort_data)
print(cohort_pivot)

    CohortMonth  CohortIndex  Customer ID
0       2009-12            0          952
1       2009-12            1          334
2       2009-12            2          317
3       2009-12            3          405
4       2009-12            4          360
..          ...          ...          ...
320     2011-10            1           70
321     2011-10            2           35
322     2011-11            0          192
323     2011-11            1           27
324     2011-12            0           28

[325 rows x 3 columns]
CohortIndex     0      1      2      3      4      5      6      7      8   \
CohortMonth                                                                  
2009-12      952.0  334.0  317.0  405.0  360.0  342.0  359.0  327.0  321.0   
2010-01      368.0   79.0  118.0  116.0  100.0  115.0   99.0   86.0  105.0   
2010-02      377.0   88.0   85.0  110.0   92.0   74.0   72.0  108.0   96.0   
2010-03      440.0   84.0  102.0  106.0  102.0   90.0  109.0  135.0  122.0   
2010

Converting the retention matrix from absolute customer counts to percentages, relative to each cohort's starting size.

In [ ]:
retention = cohort_pivot.divide(cohort_pivot[0], axis=0)
retention.index = retention.index.astype(str)
retention = retention * 100

## 6. Visualizations (Plotly)

Interactive heatmap of the retention matrix. `zmax=50` improves contrast in the meaningful 0-40% range, since only the month-0 column reaches 100%.

In [31]:
import plotly.express as px
fig = px.imshow(
    retention,
    labels=dict(x= 'Cohort Index', y= 'Cohort Month', color= 'Retention %'),
    title = 'Retention',
    zmax= 50
)
fig.show()

Interactive scatter plot: Recency vs. Monetary (log scale, due to a highly skewed distribution), colored by segment, sized by Frequency.

In [32]:
fig = px.scatter(
    rfm,
    x='Recency',
    y='Monetary',
    color='Segment',
    size='Frequency',
    log_y=True,
    title='RFM: Recency vs Monetary (color = segment, size = frequency)'
)
fig.show()

Bar chart showing customer count per RFM segment.

In [34]:
segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Count']

fig = px.bar(
    segment_counts,
    x='Segment',
    y='Count',
    color='Segment',
    title='Segments'
)
fig.show()

## 7. Revenue Forecast (Simple Model)

Aggregating monthly revenue using the full cleaned dataset (df_clean) — intentionally NOT filtered by Customer ID or cancellations, since order/cancellation pairs net out correctly on their own during summation. Dropping December 2011, an incomplete month (data ends Dec 9, 2011).

In [ ]:
df_clean['InvoiceMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['Price']
rev_month = df_clean.groupby('InvoiceMonth')['TotalPrice'].sum().reset_index()

print(df_clean['InvoiceDate'].max())

rev_month = rev_month[rev_month['InvoiceMonth'] != '2011-12']
print(rev_month)



2011-12-09 12:50:00
   InvoiceMonth   TotalPrice
0       2009-12   779173.950
1       2010-01   605070.312
2       2010-02   525551.256
3       2010-03   752494.681
4       2010-04   636798.802
5       2010-05   606642.750
6       2010-06   671408.860
7       2010-07   616706.690
8       2010-08   662445.290
9       2010-09   842040.741
10      2010-10  1069865.060
11      2010-11  1395967.862
12      2010-12   758767.090
13      2011-01   579064.410
14      2011-02   499746.020
15      2011-03   680087.540
16      2011-04   482553.601
17      2011-05   731524.100
18      2011-06   724505.150
19      2011-07   677395.411
20      2011-08   701781.590
21      2011-09  1012264.741
22      2011-10  1062417.890
23      2011-11  1428274.540


(24, 2)

Adding a sequential month index (0-23) as the time feature for the regression model.

In [48]:
rev_month['month_num'] = range(len(rev_month))
rev_month

,InvoiceMonth,TotalPrice,month_num
0,2009-12,779173.950,0
1,2010-01,605070.312,1
2,2010-02,525551.256,2
3,2010-03,752494.681,3
4,2010-04,636798.802,4
5,2010-05,606642.750,5
6,2010-06,671408.860,6
7,2010-07,616706.690,7
8,2010-08,662445.290,8
9,2010-09,842040.741,9


Baseline model: simple linear regression on the month index alone.

In [53]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = rev_month[['month_num']]
y = rev_month['TotalPrice']

model = LinearRegression()
model.fit(X, y)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[13398.49]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['month_num']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,6.169e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(1)


Evaluating the baseline model — R² is low (~0.14), since a straight line can't capture the seasonal spikes (e.g. November peak before the holidays).

In [70]:
y_pred = model.predict(X)
print(y_pred)

r2 = r2_score(y, y_pred)
print(r2)

mae = mean_absolute_error(y, y_pred)
print(mae)

[ 761711.527125  584808.368125  505389.645125  709032.117625
  552417.208625  661824.432125  690698.012125  639792.057625
  674854.447125  919893.748125 1058882.482125 1404862.208125
  776229.512875  599326.353875  519907.630875  723550.103375
  566935.194375  676342.417875  705215.997875  654310.043375
  689372.432875  934411.733875 1073400.467875 1419380.193875]
0.971887042097841
32785.45204166667


Adding seasonality: extracting calendar month and one-hot encoding it (dropping the first category to avoid the dummy variable trap).

In [65]:
rev_month['Month'] = rev_month['InvoiceMonth'].dt.month

month_dummies = pd.get_dummies(rev_month['Month'], prefix='Month', drop_first=True)
print(month_dummies)

    Month_2  Month_3  Month_4  Month_5  Month_6  Month_7  Month_8  Month_9  \
0     False    False    False    False    False    False    False    False   
1     False    False    False    False    False    False    False    False   
2      True    False    False    False    False    False    False    False   
3     False     True    False    False    False    False    False    False   
4     False    False     True    False    False    False    False    False   
5     False    False    False     True    False    False    False    False   
6     False    False    False    False     True    False    False    False   
7     False    False    False    False    False     True    False    False   
8     False    False    False    False    False    False     True    False   
9     False    False    False    False    False    False    False     True   
10    False    False    False    False    False    False    False    False   
11    False    False    False    False    False    False    Fals

Combining the month index and seasonal dummy variables into the final feature set.

In [66]:
X = pd.concat([rev_month[['month_num']], month_dummies], axis = 1)
print(X)
print(X.head())
print(X.shape)

    month_num  Month_2  Month_3  Month_4  Month_5  Month_6  Month_7  Month_8  \
0           0    False    False    False    False    False    False    False   
1           1    False    False    False    False    False    False    False   
2           2     True    False    False    False    False    False    False   
3           3    False     True    False    False    False    False    False   
4           4    False    False     True    False    False    False    False   
5           5    False    False    False     True    False    False    False   
6           6    False    False    False    False     True    False    False   
7           7    False    False    False    False    False     True    False   
8           8    False    False    False    False    False    False     True   
9           9    False    False    False    False    False    False    False   
10         10    False    False    False    False    False    False    False   
11         11    False    False    False

Retraining the model with seasonal features included.

In [68]:
model = LinearRegression()
model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](12,)","[ 1209.83,-80628.56,121804.09,...,463185.62,807955.52,178112.99]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](12,)","['month_num','Month_2','Month_3',...,'Month_10','Month_11','Month_12']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,5.836e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,12
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(12)


Evaluating the improved model — R² jumps to ~0.97. Note: with only 24 data points and 12 features, this carries real overfitting risk; there's no held-out validation set to confirm true predictive accuracy on unseen future months.

In [71]:
y_pred = model.predict(X)
print(y_pred)

r2 = r2_score(y, y_pred)
print(r2)

mae = mean_absolute_error(y, y_pred)
print(mae)

[ 761711.527125  584808.368125  505389.645125  709032.117625
  552417.208625  661824.432125  690698.012125  639792.057625
  674854.447125  919893.748125 1058882.482125 1404862.208125
  776229.512875  599326.353875  519907.630875  723550.103375
  566935.194375  676342.417875  705215.997875  654310.043375
  689372.432875  934411.733875 1073400.467875 1419380.193875]
0.971887042097841
32785.45204166667


Forecasting revenue for the next three months (Dec 2011, Jan 2012, Feb 2012) and sanity-checking the predictions against the same months in prior years.

In [77]:
X_future = pd.DataFrame({
    'month_num' : [24, 25, 26],
    'Month_2' : [0, 0, 1],
    'Month_3' : [0, 0, 0],
    'Month_4' : [0, 0, 0],
    'Month_5' : [0, 0, 0],
    'Month_6' : [0, 0, 0],
    'Month_7' : [0, 0, 0],
    'Month_8' : [0, 0, 0],
    'Month_9' : [0, 0, 0],
    'Month_10' : [0, 0, 0],
    'Month_11' : [0, 0, 0],
    'Month_12' : [1, 0, 0]
})
print(X_future)

future_pred = model.predict(X_future)
print(future_pred)

   month_num  Month_2  Month_3  Month_4  Month_5  Month_6  Month_7  Month_8  \
0         24        0        0        0        0        0        0        0   
1         25        0        0        0        0        0        0        0   
2         26        1        0        0        0        0        0        0   

   Month_9  Month_10  Month_11  Month_12  
0        0         0         0         1  
1        0         0         0         0  
2        0         0         0         0  
[790747.498625 613844.339625 534425.616625]
